# LINEARITY ANALYSIS

In [1]:
from src.system_discovery import SystemDiscovery, discoverNetwork
import src.constants as cn
from src.biomodels_iterator import BiomodelsIterator
from src.timecourse import Timecourse
from src.model import Model

import collections
import os
import numpy as np  # type: ignore
import pandas as pd # type: ignore
import matplotlib.pyplot as plt # type: ignore
import sbmlnetwork as sb # type: ignore
import tellurium as te  # type: ignore
from typing import Optional

# Helpers

In [ ]:
def checkFit(model_num: int, poly_degree=1, threshold=0.1, end_time: Optional[float]=None, num_point:int=100):
    print(F"Poly_degree: {poly_degree}, threshold: {threshold}")
    model = Model.makeBiomodel(model_num=model_num)
    timecourse = Timecourse(model, end_time=end_time, num_point=num_point)
    timecourse.plot()
    timecourse.timecourse_df
    sd = discoverNetwork(timecourse.timecourse_df, poly_degree=poly_degree, threshold=threshold)

In [ ]:
def plotValueCDF(values, title: str="", xlabel: str = "ordinate", is_plot: bool = True):
    sorted = np.sort(values)
    length = len(sorted)
    yv = np.array(range(length))/length
    plt.plot(sorted, yv)
    plt.xlabel(xlabel)
    plt.ylabel("fraction")
    plt.title(title)
    if not is_plot:
        plt.close()


In [ ]:
def plotCDF(evaluate_df: pd.DataFrame, column: str, title: str = "", is_plot: bool = True):
    list_strs = [v for v in evaluate_df[column] if isinstance(v, str)]
    if len(list_strs) > 0:
        values = []
        [values.extend(eval(v)) for v in list_strs]
        value_arr = np.array(values)
    else:
        value_arr = evaluate_df[column].values
    sorted_r2 = np.sort(value_arr)
    length = len(sorted_r2)
    yv = np.array(range(length))/length
    plt.plot(sorted_r2, yv)
    plt.xlabel("r-squared")
    plt.ylabel("fraction")
    plt.title(title)
    if not is_plot:
        plt.close()

#plotCDF("deg1_values", is_plot=False)
print("OK!")

# Detailed Analysis

## BioModels 8

In [ ]:
checkFit(8, num_point=1000)

## BioModels 206

In [ ]:
checkFit(206, threshold=0.001, poly_degree=1, num_point=1000, end_time=5)

## BioModels 181

In [ ]:
checkFit(181, threshold=0.1, poly_degree=1, num_point=1000)

# Analysis of fits

In [ ]:
for threshold in ["0.001", "0.01", "0.1", "1.0"]:
    path = os.path.join(cn.DATA_DIR, "evaluate_monomial_models-" + threshold + ".csv")
    df = pd.read_csv(path)
    df["deg1_term_density"] = df["deg1_num_nonzero_term"]/(df["num_species"]*(df["num_species"] + 1))
    plt.figure()
    plotCDF(df, "deg1_values", title=f"Number Models: {len(df)}, Threshold={threshold}")
    plotCDF(df, "deg1_min")
    plotCDF(df, "deg1_max")
    _ = plt.legend(["all species r2", "min r2 in model", "max r2 in model"])
    _ = plt.title(f"Number Models: {len(df)}, Threshold={threshold}")
    plt.figure()
    plotValueCDF(df["deg1_term_density"].values, xlabel="term density")

# Perturbations

## BioModels 181

In [ ]:
model = Model.makeBiomodel(model_num=181)
timecourse_df = Timecourse.makeTimecourses(
        model, num_point=1000,
        perturbation_value_fraction=[0.1, 0.2, 0.5],
        perturbation_species_fraction=[1.0],
        is_plot=True
    )
#timecourse_df

,T1,T2,T3,C1,C2,C3,value_frac,species_frac
0,6.600000,5.500000,1.100000,0.000000,0.000000,0.000000,0.1,1.0
1,6.585825,5.486933,1.103322,0.015727,0.012034,0.000622,0.1,1.0
2,6.571706,5.473912,1.106623,0.031398,0.024017,0.001248,0.1,1.0
3,6.557642,5.460937,1.109905,0.047014,0.035950,0.001877,0.1,1.0
4,6.543633,5.448008,1.113167,0.062574,0.047832,0.002510,0.1,1.0
...,...,...,...,...,...,...,...,...
2995,3.934642,2.316638,1.774978,7.221640,4.475756,1.044561,0.5,1.0
2996,3.934186,2.315953,1.774760,7.223571,4.475894,1.045035,0.5,1.0
2997,3.933733,2.315271,1.774543,7.225499,4.476030,1.045509,0.5,1.0
2998,3.933282,2.314591,1.774326,7.227426,4.476165,1.045982,0.5,1.0
